In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("D3_student_placement_prediction_dataset_2026 (1).csv")

เตรียมตัวแปร

In [2]:
features = [
    "aptitude_score",
    "coding_skill_score",
    "communication_skill_score",
    "logical_reasoning_score"
]

X = df[features]
y = df["salary_package_lpa"]

Split data

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Train Model (โมเดล Linear regression ธรรมดา)

In [4]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

Predict

In [5]:
y_pred = model.predict(X_test)

Evaluate

In [6]:
from sklearn.metrics import mean_squared_error, r2_score

print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MSE: 44.591477101804884
R2: 0.014524680560558112


** MSE = ความผิดพลาดเฉลี่ยกำลัง 2
ตีความคู่กับ R2 ตีความ R2 ได้ว่าโมเดลอธิบายข้อมูลได้แค่ 1.45%

ทดสอบโมเดล Polynomial Regression

In [7]:
# 1. Import
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


# 2. เลือก Feature และ Target
features = [
    "aptitude_score",
    "coding_skill_score",
    "communication_skill_score",
    "logical_reasoning_score"
]

X = df[features]
y = df["salary_package_lpa"]


# 3. Split Data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# 4. สร้างโมเดล (Polynomial + Scaling + Linear)
model = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=2),   # ลองเปลี่ยน 2,3,4 ได้
    LinearRegression()
)


# 5. Train (สำคัญมาก)
model.fit(X_train, y_train)


# 6. Predict
y_pred = model.predict(X_test)


# 7. Evaluate
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MSE: 44.59577438947281
R2: 0.014429710148889607


In [8]:
for d in [2, 3, 4]:
    model = make_pipeline(
        StandardScaler(),
        PolynomialFeatures(degree=d),
        LinearRegression()
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"Degree {d} → R2:", r2_score(y_test, y_pred))

Degree 2 → R2: 0.014429710148889607
Degree 3 → R2: 0.014244831581906703
Degree 4 → R2: 0.013506718189605915


ทดสอบโมเดลใหม่

In [9]:
# ======================
# 1. Import
# ======================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error


# ======================
# 2. Load Data
# ======================
df = pd.read_csv("D3_student_placement_prediction_dataset_2026 (1).csv")


# ======================
# 3. Feature Engineering (เพิ่มของใหม่ให้โมเดลฉลาดขึ้น)
# ======================
df["total_skill"] = (
    df["aptitude_score"] +
    df["coding_skill_score"] +
    df["communication_skill_score"] +
    df["logical_reasoning_score"]
)

df["avg_skill"] = df[[
    "aptitude_score",
    "coding_skill_score",
    "communication_skill_score",
    "logical_reasoning_score"
]].mean(axis=1)


# ======================
# 4. เลือก Feature
# ======================
features = [
    "aptitude_score",
    "coding_skill_score",
    "communication_skill_score",
    "logical_reasoning_score",
    "total_skill",
    "avg_skill"
]

X = df[features]
y = df["salary_package_lpa"]


# ======================
# 5. Split Data
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# ======================
# 6. สร้างโมเดลหลายแบบ
# ======================
models = {
    "Linear": make_pipeline(
        StandardScaler(),
        LinearRegression()
    ),

    "Polynomial (deg2)": make_pipeline(
        StandardScaler(),
        PolynomialFeatures(2),
        LinearRegression()
    ),

    "Random Forest": RandomForestRegressor(n_estimators=100),

    "Gradient Boosting": GradientBoostingRegressor()
}


# ======================
# 7. Train + Evaluate
# ======================
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("======", name, "======")
    print("R2:", r2_score(y_test, y_pred))
    print("MSE:", mean_squared_error(y_test, y_pred))
    print()

====== Linear ======
R2: 0.014524680560558112
MSE: 44.591477101804884

====== Polynomial (deg2) ======
R2: 0.014429710148889607
MSE: 44.59577438947281

====== Random Forest ======
R2: -0.047334734904011855
MSE: 47.3905352352835

====== Gradient Boosting ======
R2: 0.012341506549134573
MSE: 44.690262887729006



ทดสอบโมเดลแบบกำหนดเอง

In [10]:
new_data = pd.DataFrame({
    "aptitude_score": [60, 90],
    "coding_skill_score": [65, 95],
    "communication_skill_score": [70, 85],
    "logical_reasoning_score": [75, 92]
})

new_data["total_skill"] = new_data.sum(axis=1)
new_data["avg_skill"] = new_data.mean(axis=1)

prediction = model.predict(new_data)
print(prediction)

[6.49772056 8.69472979]
